![torchlight](https://raw.githubusercontent.com/samadpls/torchlight/main/assets/hero.png)

# Answer PyTorch forum questions with Qwen3-8B

This notebook writes the answers used to test [torchlight-qwen3-8b](https://huggingface.co/samadpls/torchlight-qwen3-8b) against plain [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B).

The questions are the 163 test questions in [torchlight-threads](https://huggingface.co/datasets/samadpls/torchlight-threads) asked from April 2025 on. Qwen3's training data reportedly runs to March 2025, so neither model has seen them. Each question's thread has a known answer that solved it.

**Links**

- Training: [the training notebook](https://www.kaggle.com/code/samadpls/torchlight-qwen3-8b-train)
- All the code, including how the answers are graded: [github.com/samadpls/torchlight](https://github.com/samadpls/torchlight)

## How it works

- The same system prompt, greedy decoding and 1,024-token limit for both models.
- Qwen3's thinking mode is off, as in training.
- Questions are sorted by length and batched, so short ones don't wait on long ones.
- The answers go to `answers-<ARM>.jsonl`, one per line.

## Before you run it

Turn on **GPU T4 x2** and **Internet** in the notebook settings. Set `ARM` below: `"Q"` for Qwen3-8B, `"QF"` for the fine-tuned model. No secrets are needed.

## Settings

In [ ]:
import json
import time

ARM = "QF"
MODELS = {"Q": "Qwen/Qwen3-8B", "QF": "samadpls/torchlight-qwen3-8b"}
# Qwen3's training data reportedly runs to March 2025, so only later threads are a fair test.
SINCE = "2025-04-01"
MAX_NEW = 1024
BUDGET = 24_576
SYSTEM = (
    "You are a careful PyTorch helper. Give specific, correct guidance with code where it helps. "
    "If a question lacks the details needed to answer it, ask for them, for example a minimal "
    "executable code snippet or the full error message, instead of guessing."
)

In [ ]:
%pip install -q -U "transformers>=4.51" accelerate datasets

## Load the questions and the model

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

test = load_dataset("samadpls/torchlight-threads", "eval", split="test")
rows = [r for r in test if r["created_at"] >= SINCE]
print(len(rows), "questions since", SINCE)

tok = AutoTokenizer.from_pretrained(MODELS[ARM])
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODELS[ARM], dtype=torch.float16, device_map="auto")
model.eval()

## Answer

In [ ]:
def prompt(question):
    chat = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    return tok.apply_chat_template(
        chat, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )


prompts = [prompt(r["question"]) for r in rows]
lengths = [len(tok(p).input_ids) for p in prompts]
order = sorted(range(len(rows)), key=lambda i: lengths[i])

start, i = time.time(), 0
with open(f"/kaggle/working/answers-{ARM}.jsonl", "w") as out:
    while i < len(order):
        # Batches pad to their longest prompt, so long prompts run in smaller batches.
        size = max(1, min(16, BUDGET // (lengths[order[i]] + MAX_NEW)))
        longest = max(lengths[j] for j in order[i : i + size])
        size = max(1, min(size, BUDGET // (longest + MAX_NEW)))
        batch = order[i : i + size]
        enc = tok([prompts[j] for j in batch], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(
                **enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id
            )
        for j, seq in zip(batch, gen[:, enc.input_ids.shape[1] :], strict=True):
            ids = [t for t in seq.tolist() if t != tok.pad_token_id]
            record = {
                "arm": ARM,
                "thread_id": rows[j]["thread_id"],
                "answer": tok.decode(ids, skip_special_tokens=True).strip(),
                "searches": [],
                "generated_tokens": len(ids),
            }
            out.write(json.dumps(record) + "\n")
        out.flush()
        i += len(batch)
        print(f"{i}/{len(rows)} answered, {time.time() - start:.0f} s")
print("done")

## A few answers

In [ ]:
by_id = {r["thread_id"]: r for r in rows}
with open(f"/kaggle/working/answers-{ARM}.jsonl") as f:
    answers = [json.loads(line) for line in f]
for a in answers[:3]:
    print("QUESTION:", by_id[a["thread_id"]]["question"][:600])
    print()
    print("ANSWER:", a["answer"])
    print("-" * 80)

## Results

Claude Opus 5.5, as judge, graded each answer from 1 to 5 against the post that solved the thread, without knowing which model wrote it.

| Model | Average grade (out of 5) | Graded 4 or 5 | Wrong answers (graded 1) |
|---|---|---|---|
| Qwen3-8B (base model) | 2.17 | 17% | 31% |
| torchlight-qwen3-8b (fine-tuned) | 2.25 | 9% | 15% |

Fine-tuning cut the wrong answers in half. Plain Qwen3-8B gives more complete fixes, mostly because it writes much longer answers: its median answer hits the 1,024-token limit, while the fine-tuned model's is 42 tokens. The [evaluation](https://github.com/samadpls/torchlight/tree/main/src/torchlight/eval) has the details and the grading prompt.

## Credits

Made by [Abdul Samad](https://samadpls.com) (abdulsamadsid1@gmail.com). The answers come from the PyTorch forum community. The dataset is licensed CC BY 4.0, and the model Apache-2.0, like Qwen3-8B. Everything else, from the forum crawl to the evaluation, is on [GitHub](https://github.com/samadpls/torchlight).